# Meta-Black-Box Optimization [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxencefaldor/bbobax/blob/main/notebooks/03_meta_bbo.ipynb)

In this notebook, we show how to do meta-black-box optimization on bbobax using Evolution Strategies.

## Install

You will need Python 3.11 or later, and a working JAX installation. For example, you can install JAX with:

In [ ]:
%pip install -U "jax[cuda]"

Then, install bbobax from PyPi:

In [ ]:
%pip install -U "bbobax[notebooks]"

## Import

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from evosax.algorithms import algorithms

import bbobax

## Build the suite

A bbobax task is **one function at one dimension** — the way COCO enumerates
problems. To meta-optimize across functions you hold many tasks and loop over
them, which is both faithful to that structure and faster than dispatching
inside a single task.

In [ ]:
fn_names = [
    "sphere",
    "ellipsoidal",
    "rastrigin",
    "bueche_rastrigin",
    "linear_slope",
    "attractive_sector",
    "step_ellipsoidal",
    "rosenbrock",
    "rosenbrock_rotated",
    "ellipsoidal_rotated",
    "discus",
    "bent_cigar",
    "sharp_ridge",
    "different_powers",
    "rastrigin_rotated",
    "weierstrass",
    "schaffers_f7",
    "schaffers_f7_ill_conditioned",
    "griewank_rosenbrock",
    "katsuura",
    "lunacek",
]

num_dims = 16

# One task per function, as COCO enumerates them. Looping over the suite keeps
# every task's function statically known: no dispatch, and each task compiles
# its own code. A single task switching over functions would, under vmap with a
# varying function, have to evaluate every branch for every solution.
tasks = bbobax.suite(fn_names, num_dims=num_dims)

## ES comparison across the suite

Each task compiles separately and instances within a task are vmapped. Scores
are averaged over instances first, then over functions, so every function
contributes equally regardless of how many instances it drew.

In [ ]:
num_generations = 512
population_size = 128
num_instances = 8  # instances per function
key = jax.random.key(0)

es_dict = {
    "SimpleES": {},
    "PGPE": {},
    "Open_ES": {"optimizer": optax.adam(1e-3)},
    "SNES": {},
    "Sep_CMA_ES": {},
    "CMA_ES": {},
}

# A dummy solution, only for shapes: every task shares the dimension.
key, subkey = jax.random.split(key)
x = next(iter(tasks.values())).sample_x(subkey)

results = {}

for es_name, es_kwargs in es_dict.items():
    print(f"Running {es_name}...")
    es = algorithms[es_name](population_size=population_size, solution=x, **es_kwargs)
    es_params = es.default_params

    def make_runner(task, es=es, es_params=es_params):
        """Build the per-instance run for one task.

        The task is closed over rather than passed, so jit sees only arrays.
        """

        def run_instance(key):
            key_task, key_solution, key_es, key_scan = jax.random.split(key, 4)
            params = task.sample(key_task)
            state = task.init(params)
            es_state = es.init(key_es, task.sample_x(key_solution), es_params)

            def step(carry, key):
                es_state, state = carry
                key_ask, key_eval, key_tell = jax.random.split(key, 3)

                population, es_state = es.ask(key_ask, es_state, es_params)
                population = jnp.clip(population, *task.x_range)

                state, evaluation = jax.vmap(task.evaluate, in_axes=(0, 0, None, None))(
                    jax.random.split(key_eval, population_size),
                    population,
                    state,
                    params,
                )
                state = jax.tree.map(lambda leaf: leaf[0], state)

                es_state, metrics = es.tell(
                    key_tell, population, evaluation.fitness, es_state, es_params
                )
                return (es_state, state), metrics

            _, metrics = jax.lax.scan(
                step, (es_state, state), jax.random.split(key_scan, num_generations)
            )
            return metrics

        return run_instance

    # Every task is its own compiled program; instances within a task vmap.
    per_task = []
    for task in tasks.values():
        key, subkey = jax.random.split(key)
        run = jax.jit(jax.vmap(make_runner(task)))
        per_task.append(run(jax.random.split(subkey, num_instances)))

    # Mean over instances, then over functions: every function counts once,
    # however easy or hard it happens to be.
    def mean_over_functions(*leaves):
        """Mean over instances first, then over functions."""
        return jnp.mean(jnp.stack([leaf.mean(axis=0) for leaf in leaves]), axis=0)

    results[es_name] = jax.tree.map(mean_over_functions, *per_task)

## Visualize

In [ ]:
# Plot the best fitness over generations for each ES
plt.figure(figsize=(10, 6))

for es_name, metrics in results.items():
    plt.plot(metrics["best_fitness"], label=es_name)

plt.title(f"Evolution strategies over the {len(tasks)}-function suite at D={num_dims}")
plt.xlabel("Generations")
plt.ylabel("Best fitness")
plt.yscale("log")
plt.legend()
plt.grid(True)
plt.show()